# Stage 2: Task-Specific AQG Training

Notebook ini untuk melatih model pada task AQG spesifik (Stage 2).

**Prerequisites:** Jalankan `02_domain_adaptation.ipynb` terlebih dahulu.

**Expected Results:**
- BLEU-4: ~0.10 → ~0.35-0.45
- ROUGE-L: ~0.40-0.50
- BERTScore F1: ~0.75-0.85
- Output model: `indot5-python-aqg`
- Training time: ~1-2 jam pada T4 GPU

## 1. Setup Environment

In [ ]:
!pip install -q transformers>=4.35.0 peft>=0.7.0 datasets>=2.15.0 accelerate>=0.25.0
!pip install -q evaluate rouge_score bert_score
print('✓ Dependencies installed')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted')

In [ ]:
import os, sys

PROJECT_ROOT = '/content/drive/MyDrive/AQG'  # UBAH SESUAI PATH ANDA
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU not available! Go to Runtime > Change runtime type > T4 GPU')

print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
print(f'✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Load Stage 1 Checkpoint

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

MODEL_NAME = 'LazarusNLP/IndoNanoT5-base'
STAGE1_MODEL_PATH = '/content/drive/MyDrive/AQG/checkpoints/domain/indot5-python-domain'

print(f'Loading Stage 1 checkpoint from: {STAGE1_MODEL_PATH}')

# Load base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load LoRA adapters dari Stage 1
peft_model = PeftModel.from_pretrained(base_model, STAGE1_MODEL_PATH)

print('✓ Stage 1 model loaded successfully')

# Print parameter info
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in peft_model.parameters())
print(f'Trainable params: {trainable:,} ({100*trainable/total:.2f}%)')

## 3. Load Task-Specific Dataset

In [ ]:
from src.finetuned.data.dataset_loader import DatasetLoader

loader = DatasetLoader()

TASK_DIR = 'dataset_aqg/dataset-task-spesifc/'

train_dataset = loader.load_dataset(TASK_DIR, split='train')
val_dataset   = loader.load_dataset(TASK_DIR, split='validation')
test_dataset  = loader.load_dataset(TASK_DIR, split='test')

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
# Validate dataset
validation_results = loader.validate_dataset(train_dataset)

# Preview sample
sample = train_dataset[0]
print('=== Sample AQG Entry ===')
print(f"Format: {sample.get('metadata', {}).get('format', 'N/A')}")
print(f"Input: {sample['input'][:300]}...")
print(f"Target: {sample['target'][:300]}...")

## 4. Baseline Evaluation (Pre-Training)

In [ ]:
from src.finetuned.evaluation.metrics_calculator import MetricsCalculator
from src.finetuned.evaluation.model_evaluator import ModelEvaluator

metrics_calc = MetricsCalculator()
evaluator = ModelEvaluator(
    model=peft_model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc
)

print('Computing baseline metrics (10 samples)...')
baseline_metrics = evaluator.evaluate_on_test_set(
    test_dataset=val_dataset,
    num_beams=4,
    include_bertscore=False,
    max_samples=10
)

print(f"\nBaseline BLEU-4: {baseline_metrics.get('bleu_4', 0):.4f}")
print(f"Baseline ROUGE-L: {baseline_metrics.get('rouge_l', 0):.4f}")

## 5. Configure Training

In [ ]:
from src.finetuned.training.task_trainer import TaskSpecificTrainer

CHECKPOINT_DIR = '/content/drive/MyDrive/AQG/checkpoints/aqg'

trainer = TaskSpecificTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    output_dir=CHECKPOINT_DIR,
    max_length=512,
    metrics_calculator=metrics_calc
)

# Training arguments sesuai spec
training_args = trainer.get_training_args(
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=30,
    logging_steps=50,
    fp16=True
)

print('Training configuration ready')
print(f'Metric for best model: {training_args.metric_for_best_model}')

## 6. Start Training

> ⚠️ Training akan memakan waktu ~1-2 jam. Checkpoints disimpan ke Google Drive.

In [ ]:
import time
start_time = time.time()

print('Starting task-specific AQG training...')
print('Checkpoints will be saved to:', CHECKPOINT_DIR)
print('='*60)

results = trainer.train(
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    training_args=training_args,
    early_stopping=True,
    early_stopping_patience=2
)

elapsed = (time.time() - start_time) / 3600
print(f'\nTraining completed in {elapsed:.2f} hours')
print(f'Final training loss: {results["training_loss"]:.4f}')

## 7. Save Final Model

In [ ]:
model_path = trainer.save_final_model(output_name='indot5-python-aqg')
print(f'Final model saved to: {model_path}')

In [ ]:
# Plot training curves
trainer.plot_training_curves(
    save_path=f'{CHECKPOINT_DIR}/training_curves.png'
)

## 8. Comprehensive Evaluation

In [ ]:
# Re-initialize evaluator dengan model yang sudah ditraining
evaluator_final = ModelEvaluator(
    model=peft_model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc
)

print('Running comprehensive evaluation on test set...')
final_metrics = evaluator_final.evaluate_on_test_set(
    test_dataset=test_dataset,
    num_beams=4,
    include_bertscore=True,
    max_samples=100
)

## 9. Generate Sample Outputs

In [ ]:
EVAL_DIR = '/content/drive/MyDrive/AQG/evaluation_results'

samples = evaluator_final.generate_samples(
    test_dataset=test_dataset,
    num_samples=20,
    num_beams=4,
    save_path=f'{EVAL_DIR}/sample_outputs.json'
)

print(f'\n✓ {len(samples)} sample outputs generated and saved')

## 10. Compare with Baseline

In [ ]:
comparison = evaluator_final.compare_with_baseline(
    finetuned_metrics=final_metrics,
    baseline_metrics=baseline_metrics
)

## 11. Final Summary

In [ ]:
import json
from pathlib import Path

# Save evaluation report
Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)
report = {
    'baseline_metrics': baseline_metrics,
    'final_metrics': final_metrics,
    'comparison': comparison,
    'training_time_hours': elapsed,
    'model_path': model_path
}
with open(f'{EVAL_DIR}/evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

# Print final summary
print('\n' + '='*60)
print('TASK-SPECIFIC AQG TRAINING SUMMARY')
print('='*60)
print(f'Training Time: {elapsed:.2f} hours')
print(f'Model saved: {model_path}')
print(f'\nMetrics Comparison:')
print(f"  BLEU-4:       {baseline_metrics.get('bleu_4',0):.4f} → {final_metrics.get('bleu_4',0):.4f}")
print(f"  ROUGE-L:      {baseline_metrics.get('rouge_l',0):.4f} → {final_metrics.get('rouge_l',0):.4f}")
print(f"  BERTScore F1: {baseline_metrics.get('bertscore_f1',0):.4f} → {final_metrics.get('bertscore_f1',0):.4f}")

bleu_improvement = comparison.get('bleu_4_improvement_pct', 0)
print(f'\nBLEU-4 Improvement: {bleu_improvement:+.1f}%')

if final_metrics.get('bleu_4', 0) >= 0.35:
    print('\n✓ SUCCESS: BLEU-4 target achieved (>= 0.35)')
else:
    print(f"\n⚠ BLEU-4 = {final_metrics.get('bleu_4',0):.4f} (target: >= 0.35)")
    print('  Consider: more epochs, lower lr, or larger dataset')

print('\n✓ Fine-tuning pipeline complete!')
print(f'  Evaluation report: {EVAL_DIR}/evaluation_report.json')
print(f'  Sample outputs: {EVAL_DIR}/sample_outputs.json')

## 12. Download Model (Optional)

In [ ]:
# Zip dan download model jika diperlukan
import shutil

zip_path = '/content/indot5-python-aqg.zip'
shutil.make_archive('/content/indot5-python-aqg', 'zip', model_path)

from google.colab import files
files.download(zip_path)
print(f'✓ Model downloaded: {zip_path}')